In [1]:
!pip install langchain-openai langchain pdfminer.six chromadb rank_bm25 langchain-community langchain-text-splitters

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
except ImportError:
    pass  # not running in Colab -- assume OPENAI_API_KEY is already set in the environment

from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from pdfminer.high_level import extract_text as extract_text_pdf_miner
from langchain_community.vectorstores import Chroma
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

C:\Users\Avado\AppData\Local\Temp\ipykernel_40928\1848288847.py:15: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


## Hybrid Search

In [3]:
# Define the directory where the Chroma database will persist data
persist_directory = "/content/hybrid-search"

# Initialize OpenAI embeddings with the specified model
# "text-embedding-3-small" is OpenAI's current, cost-efficient embedding model
embedding = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [4]:
def load_data_to_vectordb(file_path,source):
  global bm25_retriever
  # Loop through a list of PDF files to process
  pages = []

  for pdf_name in [file_path]:
      # Open each PDF file in binary mode
      with open(pdf_name, 'rb') as f:
          # Extract text from the PDF using the extract_text_pdf_miner function
          text = extract_text_pdf_miner(f)

          # Clean the extracted text by removing newline characters and joining into a single string
          cleaned_text = " ".join(text.split("\n"))

          # Initialize a list to store document chunks
          docs = []

          # Create a text splitter to divide the text into manageable chunks
          # Each chunk has a maximum size of 2048 characters with a 512-character overlap
          splitter = RecursiveCharacterTextSplitter(chunk_size=2048, chunk_overlap=512)

          # Split the cleaned text into chunks and wrap each chunk in a Document object
          for chunk in splitter.split_text(cleaned_text):
              docs.append(Document(page_content=chunk, metadata={"retrived_from":source,"source": pdf_name}))
              pages.append(Document(page_content=chunk, metadata={"retrived_from":source,"source": pdf_name}))
      # Create a Chroma collection from the processed documents
      # Use the specified persist directory and embedding model for storage and retrieval
      if source == 1:
        bm25_retriever = BM25Retriever.from_documents(pages)
      else:
        db = Chroma.from_documents(
            documents=docs,
            persist_directory=persist_directory,
            embedding=embedding
        )

load_data_to_vectordb(file_path="/content/1706.03762v7.pdf",source=1)
load_data_to_vectordb(file_path="/content/1506.02640v5.pdf",source=2)

In [5]:
# Initialize the BM25 retriever
bm25_retriever.k = 2  # Retrieve top 2 results
print("type of bm25", type(bm25_retriever))

type of bm25 <class 'langchain_community.retrievers.bm25.BM25Retriever'>


In [6]:
# Initialize retriever
docsearch = Chroma(persist_directory=persist_directory, embedding_function=embedding)
retriever_chromadb = docsearch.as_retriever(search_kwargs={"k": 5})

# Initialize the ensemble retriever
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, retriever_chromadb], weights=[0.3, 0.7]
)

C:\Users\Avado\AppData\Local\Temp\ipykernel_40928\3235932638.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  docsearch = Chroma(persist_directory=persist_directory, embedding_function=embedding)


In [7]:
# Example query
query = "What is Self-Attention in Transformers?"

# Retrieve relevant documents/products
docs = ensemble_retriever.invoke(query)

docs

[Document(metadata={'retrived_from': 2, 'source': '/content/1506.02640v5.pdf'}, page_content='features. In Computer vision, 1999. The proceedings of the seventh IEEE international conference on, volume 2, pages 1150–1157. Ieee, 1999. 4  [24] D. Mishkin.  Models accuracy on imagenet 2012 https://github.com/BVLC/caffe/wiki/ val. Models-accuracy-on-ImageNet-2012-val. Ac- cessed: 2015-10-2. 3  [25] C. P. Papageorgiou, M. Oren, and T. Poggio. A general framework for object detection. In Computer vision, 1998. sixth international conference on, pages 555–562. IEEE, 1998. 4  [26] J. Redmon. Darknet: Open source neural networks in c. http://pjreddie.com/darknet/, 2013–2016. 3 [27] J. Redmon and A. Angelova. Real-time grasp detection using convolutional neural networks. CoRR, abs/1412.3128, 2014. 5  [28] S. Ren, K. He, R. Girshick, and J. Sun. Faster r-cnn: To- wards real-time object detection with region proposal net- works. arXiv preprint arXiv:1506.01497, 2015. 5, 6, 7 [29] S. Ren, K. He, R.

In [8]:
#Extract and print only the page content from each document
import pandas as pd

retrieval_df = pd.DataFrame()

page_content = []
retrieval_source = []
pdf_source = []

for doc in docs:
    page_content.append(doc.page_content)
    retrieval_source.append(doc.metadata['retrived_from'])
    pdf_source.append(doc.metadata['source'])

retrieval_df['page_content'] = page_content
retrieval_df['retrieval_source'] = retrieval_source
retrieval_df['pdf_source'] = pdf_source

retrieval_df.head(10)

,page_content,retrieval_source,pdf_source
0,"features. In Computer vision, 1999. The procee...",2,C:/Users/Avado/AppData/Local/Temp/claude/C--Us...
1,localiza- tion and detection using convolution...,2,C:/Users/Avado/AppData/Local/Temp/claude/C--Us...
2,want one bounding box predictor to be responsi...,2,C:/Users/Avado/AppData/Local/Temp/claude/C--Us...
3,identically. This consists of two linear trans...,1,C:/Users/Avado/AppData/Local/Temp/claude/C--Us...
4,our model contains no recurrence and no convol...,1,C:/Users/Avado/AppData/Local/Temp/claude/C--Us...


## Re Ranking

In [9]:
# Define the directory where the Chroma database will persist data
persist_directory = "/content/re-rank"

# Initialize OpenAI embeddings with the specified model
# "text-embedding-3-small" is OpenAI's current, cost-efficient embedding model
embedding = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [10]:
def load_data_to_vectordb(file_path,source):
  # Loop through a list of PDF files to process

  for pdf_name in [file_path]:
      # Open each PDF file in binary mode
      with open(pdf_name, 'rb') as f:
          # Extract text from the PDF using the extract_text_pdf_miner function
          text = extract_text_pdf_miner(f)

          # Clean the extracted text by removing newline characters and joining into a single string
          cleaned_text = " ".join(text.split("\n"))

          # Initialize a list to store document chunks
          docs = []

          # Create a text splitter to divide the text into manageable chunks
          # Each chunk has a maximum size of 2048 characters with a 512-character overlap
          splitter = RecursiveCharacterTextSplitter(chunk_size=2048, chunk_overlap=512)

          # Split the cleaned text into chunks and wrap each chunk in a Document object
          for chunk in splitter.split_text(cleaned_text):
              docs.append(Document(page_content=chunk, metadata={"source": pdf_name}))
      # Create a Chroma collection from the processed documents
      # Use the specified persist directory and embedding model for storage and retrieval

      db = Chroma.from_documents(
            documents=docs,
            persist_directory=persist_directory,
            embedding=embedding
        )

load_data_to_vectordb(file_path="/content/1706.03762v7.pdf",source=1)
load_data_to_vectordb(file_path="/content/1506.02640v5.pdf",source=2)

In [11]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMListwiseRerank

In [12]:
docsearch = Chroma(persist_directory=persist_directory, embedding_function=embedding)

In [13]:
# Initialize an empty DataFrame with specified columns
non_rerank_df = pd.DataFrame(columns=['Text', 'source', 'relevance_score'])

# Perform similarity search using a preconfigured document search tool
# This retrieves the top 3 documents based on relevance to the query
res_docs = docsearch.similarity_search_with_relevance_scores("What is the architecture of transformers?", k=3)

# Loop through the retrieved documents and populate the DataFrame
# (DataFrame._append was removed in pandas 3.x -- pd.concat is the modern, version-safe way to add a row)
for doc in res_docs:
    new_row = pd.DataFrame([{
        'Text': doc[0].page_content,          # Extract the page content (text) from the document
        'source': doc[0].metadata['source'],  # Extract the source metadata
        'relevance_score': doc[1]             # Extract the relevance score
    }])
    non_rerank_df = pd.concat([non_rerank_df, new_row], ignore_index=True)

# Display the first 3 rows of the DataFrame
non_rerank_df.head(3)

,Text,source,relevance_score
0,mechanism instead of sequence- aligned recurre...,C:/Users/Avado/AppData/Local/Temp/claude/C--Us...,0.340344
1,mechanism instead of sequence- aligned recurre...,C:/Users/Avado/AppData/Local/Temp/claude/C--Us...,0.340344
2,global dependencies between input and output. ...,C:/Users/Avado/AppData/Local/Temp/claude/C--Us...,0.286772


In [14]:
# Import and initialize the reranker for document compression
# OpenAI does not offer a standalone hosted rerank endpoint like Cohere's CohereRerank,
# so LLMListwiseRerank is used instead: it reranks candidate documents by asking a chat
# model (here, gpt-4o-mini) to directly judge and reorder them by relevance -- the same
# core idea as RankGPT.
compressor = LLMListwiseRerank.from_llm(llm=ChatOpenAI(model="gpt-4o-mini", temperature=0), top_n=3)
# LLMListwiseRerank is a document compressor that uses an LLM to rerank documents by relevance.

# Create a ContextualCompressionRetriever for improved document retrieval
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,               # Use the reranker as the base compression mechanism
    base_retriever=docsearch.as_retriever()   # Use the existing document search tool as the base retriever
)
# The ContextualCompressionRetriever combines the base retriever's results with reranking
# to provide more contextually relevant and concise results.

In [15]:
# Initialize an empty DataFrame with specified columns
source_df = pd.DataFrame(columns=['Text', 'source', 'rerank_position'])

# Retrieve compressed (reranked) documents relevant to the query using the contextual compression retriever
compressed_docs = compression_retriever.invoke("What is the architecture of Transformers?")

# Loop through the reranked documents and populate the DataFrame
# (DataFrame._append was removed in pandas 3.x -- pd.concat is the modern, version-safe way to add a row)
for i, doc in enumerate(compressed_docs):
    new_row = pd.DataFrame([{
        'Text': doc.page_content,          # Extract the content of the document
        'source': doc.metadata['source'],   # Extract the source information
        'rerank_position': i + 1            # LLMListwiseRerank returns documents already in relevance order (1 = most relevant)
    }])
    source_df = pd.concat([source_df, new_row], ignore_index=True)

# Display the first 3 rows of the DataFrame
source_df.head(3)

,Text,source,rerank_position
0,mechanism instead of sequence- aligned recurre...,C:/Users/Avado/AppData/Local/Temp/claude/C--Us...,1
1,mechanism instead of sequence- aligned recurre...,C:/Users/Avado/AppData/Local/Temp/claude/C--Us...,2
2,global dependencies between input and output. ...,C:/Users/Avado/AppData/Local/Temp/claude/C--Us...,3


We can see the last result is different and the relevance scores are also different but they are according to the relevance model